<a href="https://colab.research.google.com/github/Gabo190594/recommender-system-supermarket/blob/main/notebooks/03_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Clonar tu repositorio desde GitHub
!git clone https://github.com/Gabo190594/recommender-system-supermarket.git

# Entrar a la carpeta del proyecto
%cd recommender-system-supermarket

# Generar data si no existe
import os

if not os.path.exists("data/raw/users.csv"):
    print("Generando data...")
    !python data_generation/generate_data.py
else:
    print("Data ya existe")

Cloning into 'recommender-system-supermarket'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 59 (delta 21), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 243.14 KiB | 4.19 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/recommender-system-supermarket
Data ya existe


In [4]:
from IPython.display import Markdown, display

display(Markdown("""
# 03. Preparación de datos

## Objetivo
Construir un dataset procesado e integrado a partir de las tablas originales.

## ¿Qué hace este notebook?
1. Une las interacciones con la información de productos y usuarios.
2. Incorpora los ratings explícitos por usuario-producto.
3. Elimina duplicados y trata valores faltantes.
4. Convierte la columna de tiempo a formato fecha.
5. Extrae variables temporales como mes y día de la semana.
6. Normaliza variables numéricas como edad, precio, score implícito y rating.

## Resultado esperado
Generar un archivo procesado con las variables necesarias para las siguientes etapas del proyecto.
"""))


# 03. Preparación de datos

## Objetivo
Construir un dataset procesado e integrado a partir de las tablas originales.

## ¿Qué hace este notebook?
1. Une las interacciones con la información de productos y usuarios.
2. Incorpora los ratings explícitos por usuario-producto.
3. Elimina duplicados y trata valores faltantes.
4. Convierte la columna de tiempo a formato fecha.
5. Extrae variables temporales como mes y día de la semana.
6. Normaliza variables numéricas como edad, precio, score implícito y rating.

## Resultado esperado
Generar un archivo procesado con las variables necesarias para las siguientes etapas del proyecto.


In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from IPython.display import Markdown, display

# =========================
# CARGA DE DATOS
# =========================
users = pd.read_csv("data/raw/users.csv")
products = pd.read_csv("data/raw/products.csv")
interactions = pd.read_csv("data/raw/interactions.csv")
ratings = pd.read_csv("data/raw/ratings.csv")
social_links = pd.read_csv("data/raw/social_links.csv")

# =========================
# UNIÓN DE DATOS
# =========================
df = interactions.merge(products, on="product_id", how="left")
df = df.merge(users, on="user_id", how="left")
df = df.merge(ratings, on=["user_id", "product_id"], how="left")

print("Shape del dataset unido:", df.shape)
display(df.head())

# =========================
# LIMPIEZA BÁSICA
# =========================
df = df.drop_duplicates()

# Completar rating faltante con la mediana
if "rating" in df.columns:
    df["rating"] = df["rating"].fillna(df["rating"].median())

# Convertir timestamp
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

# Extraer variables de tiempo
df["month"] = df["timestamp"].dt.month
df["dayofweek"] = df["timestamp"].dt.dayofweek

# =========================
# NORMALIZACIÓN MIN-MAX
# =========================
scaler = MinMaxScaler()

cols_to_normalize = ["age", "price", "implicit_score", "rating"]

for col in cols_to_normalize:
    if col in df.columns:
        df[f"{col}_norm"] = scaler.fit_transform(df[[col]])

# =========================
# RESUMEN
# =========================
display(Markdown(f"""
## ✅ Preparación completada

- Registros finales: **{len(df):,}**
- Usuarios únicos: **{df['user_id'].nunique()}**
- Productos únicos: **{df['product_id'].nunique()}**
- Columnas disponibles: **{len(df.columns)}**
"""))

# =========================
# GUARDAR
# =========================
df.to_csv("data/processed/prepared_data.csv", index=False)
print("Archivo guardado en: data/processed/prepared_data.csv")

Shape del dataset unido: (15001, 10)


,user_id,product_id,event_type,timestamp,implicit_score,category,price,age,segment,rating
0,320,343,view,2025-05-12,1,lacteos,49.18,44,familia,NaN
1,138,367,view,2025-03-04,1,snacks,36.56,31,familia,NaN
2,38,465,purchase,2025-05-12,5,frutas,80.98,54,soltero,NaN
3,111,241,view,2025-05-13,1,bebidas,12.61,56,familia,NaN
4,480,199,view,2025-05-18,1,bebidas,43.19,51,soltero,NaN



## ✅ Preparación completada

- Registros finales: **14,999**
- Usuarios únicos: **500**
- Productos únicos: **800**
- Columnas disponibles: **16**


Archivo guardado en: data/processed/prepared_data.csv
